# 🚀 Colab Ollama MCP — Remote GPU Runner

> **Mục đích:** Hướng dẫn từng bước chạy Ollama trên Google Colab GPU (T4 / V100 / A100) và mở đường hầm Cloudflare Tunnel để kết nối trực tiếp với **Claude Desktop**, **Cursor**, **Antigravity IDE** qua giao thức **MCP (Model Context Protocol)**.

---
### 📌 Lưu ý trước khi chạy:
1. Vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** (hoặc A100 nếu dùng Colab Pro).
2. Chạy lần lượt các ô (cells) từ trên xuống dưới hoặc nhấn `Runtime -> Run all`.

### Bước 1: Kiểm tra thông số GPU (VRAM & Driver)
Kiểm tra xem Colab đã cấp GPU chưa (Thường là Tesla T4 16GB VRAM trên Colab Free/Pro).

In [ ]:
!nvidia-smi

### Bước 2 (Khuyên dùng): Gắn Google Drive để lưu cache Model
> 💡 **Mẹo tối ưu:** Model LLM nặng từ 4GB đến 10GB. Nếu không mount Google Drive, mỗi lần tắt Colab bạn sẽ phải tải lại từ đầu làm tốn thời gian. Khi trỏ `OLLAMA_MODELS` vào Drive, model chỉ cần tải một lần duy nhất!

In [ ]:
USE_GDRIVE = True  # Đổi thành False nếu bạn không muốn lưu vào Google Drive

import os
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
    os.makedirs('/content/drive/MyDrive/ollama_models', exist_ok=True)
    print("✅ Đã cấu hình lưu cache Model tại: /content/drive/MyDrive/ollama_models")
else:
    print("ℹ️ Sử dụng bộ nhớ tạm của Colab (model sẽ bị xóa khi ngắt phiên).")

### Bước 3: Cài đặt Ollama & Khởi động Daemon
Cài đặt Ollama Linux runtime và khởi chạy ngầm daemon `ollama serve` lắng nghe trên cổng `11434`.

In [ ]:
import subprocess
import time
import urllib.request

# 1. Cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Khởi động Ollama daemon trong tiến trình ngầm
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Chờ daemon sẵn sàng
time.sleep(3)
try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Ollama Daemon đã khởi động thành công:", resp.read().decode('utf-8'))
except Exception as e:
    print("⚠️ Chưa kết nối được tới Ollama daemon:", e)

### Bước 4: Tải mô hình LLM mong muốn
Gợi ý các model phù hợp với GPU T4 (16GB VRAM):
- `qwen2.5-coder:7b`: Model viết code, review code hàng đầu hiện nay trong phân khúc 7B.
- `llama3.1:8b`: Model đa năng, tư duy logic, tổng hợp thông tin xuất sắc.
- `deepseek-r1:8b`: Model suy luận phân tích từng bước (Reasoning/Chain-of-thought).

In [ ]:
# Thay đổi tên model bạn muốn tải ở đây
MODEL_NAME = "qwen2.5-coder:7b"

print(f"⏳ Đang kéo mô hình {MODEL_NAME}...")
!ollama pull {MODEL_NAME}
print(f"\n✅ Đã tải xong! Danh sách model hiện có:")
!ollama list

### Bước 5: Cài đặt Cloudflare Quick Tunnel & Lấy Public URL
Cloudflare Quick Tunnel giúp tạo một đường hầm HTTPS bảo mật an toàn ra Internet mà **không cần tạo tài khoản** hay cấu hình phức tạp.

In [ ]:
import re
import time

# 1. Tải và cài đặt file binary cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Khởi chạy Cloudflare Tunnel nối tới cổng 11434 của Ollama
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Bắt link *.trycloudflare.com từ log đầu ra
tunnel_url = None
print("⏳ Đang thiết lập đường hầm Cloudflare Tunnel...")
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

if tunnel_url:
    print("=" * 70)
    print("🎉 CHÚC MỪNG! ĐƯỜNG HẦM OLLAMA ĐÃ HOẠT ĐỘNG:")
    print(f"🔗 TUNNEL URL: {tunnel_url}")
    print("=" * 70)
    print("\n📋 HƯỚNG DẪN CẤU HÌNH LOCAL MCP CLIENT:")
    print(f"Thêm đoạn cấu hình sau vào file mcp_config.json của Claude Desktop hoặc Cursor:\n")
    config_json = f'''{{
  "mcpServers": {{
    "colab-ollama": {{
      "command": "python3",
      "args": [
        "/absolute/path/to/colab_ollama_mcp/server.py",
        "--endpoint",
        "{tunnel_url}"
      ]
    }}
  }}
}}'''
    print(config_json)
else:
    print("❌ Không tìm thấy tunnel URL. Vui lòng kiểm tra lại log của cloudflared.")

### Bước 6: Giữ phiên Colab hoạt động (Anti-Idle / Keep-Alive)
Google Colab sẽ tự ngắt kết nối nếu bạn không mở tab trình duyệt sau 60-90 phút. Cell dưới đây chạy một trình phát âm thanh vô thanh nhẹ nhàng giúp trình duyệt nhận biết tab đang hoạt động.

In [ ]:
from IPython.display import HTML

HTML('''
<div style="padding: 15px; background: #e8f5e9; border: 1px solid #4caf50; border-radius: 8px;">
  <h4 style="margin: 0; color: #2e7d32;">🟢 Hệ thống Keep-Alive đang chạy ngầm</h4>
  <p style="margin: 5px 0 0 0; color: #1b5e20; font-size: 13px;">Trình duyệt sẽ phát âm thanh tần số vô thanh để ngăn Colab tự động chuyển sang chế độ ngủ.</p>
  <audio loop autoplay controls style="margin-top: 10px; width: 220px; height: 30px;">
    <source src="https://raw.githubusercontent.com/anars/blank-audio/master/500-milliseconds-of-silence.mp3" type="audio/mp3">
  </audio>
</div>
''')